In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\End-to-End-Machine_learning_project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Hp\\Desktop\\End-to-End-Machine_learning_project'

Entity

In [5]:
from dataclasses import dataclass # dataclass given deceroter
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path 
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

update the configuration manger is src config

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema["COLUMNS"]

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [8]:
import os
from mlProject import logger 
import pandas as pd


In [9]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validation_all_columns(self)-> bool:
        try:
            validation_status= None

            data = pd.read_csv(self.config.unzip_data_dir)
            
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()


            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validatiaonstatus: {validation_status}")

            return validation_status
        except Exception as e:
            raise e



pipeline

In [11]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validation_all_columns()
except Exception as e:
    raise e

[2026-04-24 11:58:53,125: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-24 11:58:53,127: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-24 11:58:53,133: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-24 11:58:53,137: INFO: common: created directory at: artifacts]
[2026-04-24 11:58:53,138: INFO: common: created directory at: artifacts/data_validation]


debug

In [ ]:
import pandas as pd
import yaml

# ✅ read yaml safely
def read_yaml(path_to_yaml: str):
    with open(path_to_yaml, "r") as file:
        return yaml.safe_load(file)

# ✅ load schema
schema = read_yaml(r"C:\Users\Hp\Desktop\End-to-End-Machine_learning_project\schema.yaml")

schema_columns = list(schema["COLUMNS"].keys())

# ✅ load data
data = pd.read_csv(r"artifacts/data_ingestion/WineQT.csv")



# ✅ validation
validation_status = set(data.columns) == set(schema_columns)

print(validation_status)

False


In [ ]:
print("DATA COLS:")
print(set(data.columns))

print("\nSCHEMA COLS:")
print(set(schema_columns))

print("\nMISSING IN DATA:")
print(set(schema_columns) - set(data.columns))

print("\nEXTRA IN DATA:")
print(set(data.columns) - set(schema_columns))

DATA COLS:
{'free sulfur dioxide', 'pH', 'residual sugar', 'chlorides', 'alcohol', 'citric acid', 'total sulfur dioxide', 'Id', 'fixed acidity', 'quality', 'volatile acidity', 'density', 'sulphates'}

SCHEMA COLS:
{'free sulfur dioxide', 'residual sugar', 'ph', 'chlorides', 'alcohol', 'citric acid', 'total sulfur dioxide', 'fixed acidity', 'quality', 'volatile acidity', 'density', 'sulphates'}

MISSING IN DATA:
{'ph'}

EXTRA IN DATA:
{'pH', 'Id'}


In [ ]:
print(data.columns)
print(schema["COLUMNS"].keys())

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality', 'Id'],
      dtype='object')
dict_keys(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'ph', 'sulphates', 'alcohol', 'quality'])
